# 02. Leak-Free Feature Engineering Pipeline
### Building Production-Grade Transformations for Hospital Scheduling

**Key Requirement:** No data leakage. We must only use information available at the exact moment of booking.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np

from src.data.load_data import load_dataset
from src.data.preprocess import clean_raw_data, prepare_train_test_data
from src.features.build_features import extract_date_features, compute_patient_history_features

df, _ = load_dataset()
clean_df = clean_raw_data(df)
print(f"Cleaned dataset: {len(clean_df):,} records.")

## 1. Temporal Feature Decomposition
Extracting calendar day-of-week, hour of booking, lead days, and age categories.

In [ ]:
feat_sample = extract_date_features(clean_df.head(100))
feat_sample[['ScheduledDay', 'AppointmentDay', 'Waiting_Days', 'Is_Same_Day', 'Scheduled_Hour', 'Appointment_Day_Of_Week', 'Age_Group']].head(5)

## 2. Chronological Patient Attendance History (No Target Leakage)
We calculate cumulative previous visits and no-shows up to (but not including) the current visit.

In [ ]:
# Verify on a patient with multiple visits
repeat_patient = clean_df['PatientId'].value_counts().index[0]
sample_patient_history = compute_patient_history_features(clean_df[clean_df['PatientId'] == repeat_patient])
sample_patient_history[['PatientId', 'ScheduledDay', 'prior_appointments', 'prior_no_shows', 'prior_no_show_rate', 'No-show']].head(5)

## 3. Fit ColumnTransformer on Training Split Only

In [ ]:
X_train, X_test, y_train, y_test, preprocessor, feature_names = prepare_train_test_data(clean_df)
print(f"Training set: {len(X_train):,} | Test set: {len(X_test):,}")
print(f"Transformed features: {len(feature_names)}")
print("Sample feature names:", feature_names[:12])